<div align="center">

# Week 10: Learning from Data: Load and Generation Forecasting, Anomaly Detection

**Cyber-Physical Systems (BIL-435), hands-on Colab notebook**

**Prof. Dr. Utku Kose**

Full Professor, Department of Computer Engineering, Süleyman Demirel University, Isparta, Türkiye  
Founding Director, AI Application and Research Center (YAZEM), Süleyman Demirel University  
Head of the Computer Science Division, Department of Computer Engineering, Süleyman Demirel University  
Additional affiliations: University of North Dakota (USA), Universidad Panamericana (Mexico City, Mexico), Vel Tech University (Chennai, India)  
IEEE Senior Member, ACM Professional Member

[ORCID 0000-0002-9652-6415](https://orcid.org/0000-0002-9652-6415) | [utkukose.com](https://www.utkukose.com) | [github.com/utkukose](https://github.com/utkukose)

[utkukose@sdu.edu.tr](mailto:utkukose@sdu.edu.tr) | [utku.kose@und.edu](mailto:utku.kose@und.edu) | [ukose@up.edu.mx](mailto:ukose@up.edu.mx) | [utkukose@gmail.com](mailto:utkukose@gmail.com)

[Week overview](https://github.com/utkukose/cyber-physical-systems-NB-lecture/tree/main/weeks/week-10) · [Lecture page](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-10/lecture.html) · [Interactive lab](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-10/lab.html) · [PDF notes](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-10/Week10_Lecture_Notes.pdf)

</div>

## How to use this Colab notebook

This Colab notebook is the hands-on part of the week. The concepts are explained on the [lecture page](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-10/lecture.html), and each section that applies a part of the lecture starts with a link to it. Save your own copy first with File, Save a copy in Drive, then run the cells in order with Shift+Enter. Exercises check your answer as soon as you run them. Each exercise has a reference solution in a collapsed cell: Try first, then open it. Running all cells at once also runs the reference solutions. Cells titled *Observe and reflect* ask open questions about what the code shows. They have no solution cell, and a short answer in a text cell of your copy (Insert, Text cell) is enough. Cells titled *Python note* explain a piece of the Python language used nearby. Nothing needs to be installed.

**Starting in the middle.** Each numbered section can also be opened on its own, for example from an orange Colab box of the lecture page. Its first code cell, *Catch-up*, runs the code of the earlier sections without their printouts and figures. When the Colab notebook runs from the top, the catch-up cells notice it and do nothing.

## 0. Setup

The first cell prepares the environment and defines the helper `check`, which compares an answer with the expected value and prints immediate feedback. It also builds the eight weeks of the campus as the DataFrame `HIST`, with the maintenance log of the solar plant. It holds the tariff and the battery of Week 9 as well. The history is synthetic, built from the models of the earlier weeks with random weather, and every run gives the same eight weeks. The second cell keeps the expected values of the exercises in the table `ANSWERS`, with its code hidden, so that no answer stands next to its exercise.

In [ ]:
# Environment: Everything used by this course is preinstalled in Google Colab.
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

import io
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import linprog

plt.rcParams.update({"figure.dpi": 110, "figure.figsize": (7, 3.6),
                     "axes.grid": True, "grid.alpha": 0.3})

def check(name, value, expected=None, tol=1e-6):
    """Compares an answer with the expected value and prints immediate feedback.
    Without an expected value in the call, check takes the value and its tolerance
    from ANSWERS, the table of the next cell."""
    if expected is None:
        answer, tol = ANSWERS[name]
        expected = answer()
    if value is None:
        print(f"{name}: not answered yet. Replace None and run the cell again.")
        return False
    try:
        scale = max(1.0, abs(float(expected)))
        ok = abs(float(value) - float(expected)) <= tol * scale
    except (TypeError, ValueError):
        ok = value == expected
    print(f"{name}: " + ("correct." if ok else
                         "not yet. Compare your steps with the lecture."))
    return ok

# The spring weekday of Week 1 on the campus, one value per hour (kW, m/s, W/m2 and degrees Celsius)
HOURS = np.arange(24)
LOAD_KW = np.array([262, 255, 250, 248, 252, 270, 330, 430, 520, 570, 585, 590,
                    575, 580, 585, 575, 560, 530, 520, 500, 450, 390, 320, 280], float)
EV_KW = np.array([0, 0, 0, 0, 0, 0, 0, 22, 66, 66, 44, 22,
                  22, 22, 22, 44, 66, 110, 110, 88, 44, 22, 0, 0], float)
WIND_MS = np.array([7.8, 8.0, 8.2, 8.0, 7.6, 7.0, 6.2, 5.4, 4.8, 4.4, 4.2, 4.6,
                    5.0, 5.2, 5.0, 4.6, 4.2, 3.8, 3.6, 3.5, 3.8, 4.6, 5.8, 7.0])
G_SKY = 950 * np.maximum(0, np.sin(np.pi * (HOURS + 0.5 - 6) / 13))
T_AMB = 12 + 7 * np.sin(np.pi * (HOURS - 9) / 12)

def pv_power(irr, temp):
    """The solar plant of 400 kW peak (Week 1): irradiance in W/m2 and air temperature in degrees Celsius."""
    return 400 * irr / 1000 * (1 - 0.004 * (temp + 0.03 * irr - 25))

def wind_power(v):
    """The power curve of the wind turbine of 250 kW (Week 1), wind speed v in m/s."""
    p = 250 * (v ** 3 - 27) / (12 ** 3 - 27)
    return np.where(v < 3, 0.0, np.where(v < 12, p, 250.0))

def cloud_factor(okta):
    """The share of the clear-sky irradiance under a cloud cover of okta eighths (Kasten and Czeplak)."""
    return 1 - 0.75 * (np.asarray(okta, float) / 8) ** 3.4

def campus_history(seed=10):
    """Eight weeks of hourly data of the campus, Monday 2 March to Sunday 26 April 2026 (a synthetic history)."""
    rng = np.random.default_rng(seed)
    days = pd.date_range("2026-03-02", periods=56, freq="D")
    free = (days.weekday >= 5) | days.isin(pd.to_datetime(["2026-03-20", "2026-04-23"]))   # weekends and holidays
    d = np.arange(56)
    w = np.zeros(56)
    for k in range(1, 56):
        w[k] = 0.7 * w[k - 1] + rng.normal(0, 1.8)                                         # the weather of each day
    cold = np.select([np.isin(d, [44, 45, 46]), d == 47], [-7.0, -3.5], 0.0)              # a cold spell, 15 to 18 April
    temp = (5.5 + 8 * d / 55 + w + cold)[:, None] + 6.5 * np.sin(np.pi * (HOURS - 9) / 12) + rng.normal(0, 0.4, (56, 24))
    temp_fc = temp + rng.normal(0, 1.2, (56, 1)) + rng.normal(0, 0.3, (56, 24))
    heat = 5.5 * np.maximum(0, 15 - temp)                                                   # heating, kW per kelvin below 15 C
    work = LOAD_KW - 5.5 * np.maximum(0, 15 - T_AMB)
    base = np.where(free[:, None], 0.55 * work + 0.45 * work.min(), work)
    load = (base + heat) * (1 + 0.02 * rng.normal(0, 1, (56, 1)) + 0.025 * rng.normal(0, 1, (56, 24)))
    ev = np.where(free[:, None], 0.0, EV_KW)
    u = rng.uniform(0, 1, 56)
    p_clear = np.where(d < 28, 0.30, 0.45)
    kind = np.where(u < p_clear, 0, np.where(u < p_clear + 0.35, 1, 2))                    # clear, broken, overcast
    okta = np.where(kind == 0, rng.uniform(0, 2, 56), np.where(kind == 1, rng.uniform(3, 5.5, 56), rng.uniform(6, 8, 56)))
    okta_h = np.clip(okta[:, None] + np.where(kind == 1, 1.6, 0.5)[:, None] * rng.normal(0, 1, (56, 24)), 0, 8)
    clear = (0.92 + 0.08 * d / 55)[:, None] * G_SKY
    irr = clear * cloud_factor(okta_h)
    pv = pv_power(irr, temp) * (1 + 0.012 * rng.normal(0, 1, (56, 24)))
    irr_meas = irr * (1 + 0.008 * rng.normal(0, 1, (56, 24)))
    okta_fc = np.clip(np.round(okta + rng.normal(0, 1.3, 56)), 0, 8)
    v = WIND_MS * np.exp(rng.normal(0, 0.35, (56, 1)))
    v_fc = v * np.exp(rng.normal(0, 0.15, (56, 1)))
    wind = wind_power(v)
    # three faults of the solar plant, as recorded in the maintenance log
    fault = np.zeros((56, 24), bool)
    pv[43, 10:14] *= 0.5                                       # inverter 2 trips on 14 April, 10:00 to 13:59
    pv[49:] *= (1 - 0.008 * (d[49:] - 48))[:, None]            # dust from 20 April: 0.8 percent less output each day
    pv_meter = pv.copy()
    pv_meter[47, 13:17] = pv_meter[47, 12]                     # the meter of the plant freezes on 18 April from 13:00
    fault[43, 10:14] = fault[47, 13:17] = True
    fault[49:] = True
    grid = (load + ev - pv - wind).round(1)                    # the grid meter sees the real power of the plant
    index = pd.date_range("2026-03-02", periods=56 * 24, freq="h")
    flat = lambda a: np.asarray(a, float).ravel()
    hist = pd.DataFrame({"load_kw": flat(load).round(1), "ev_kw": flat(ev), "pv_kw": flat(pv_meter).round(1),
                         "wind_kw": flat(wind).round(1), "grid_kw": flat(grid), "temp_c": flat(temp).round(1),
                         "irr_wm2": flat(irr_meas).round(1), "irr_clear_wm2": flat(clear).round(1),
                         "temp_fc_c": flat(temp_fc).round(1), "cloud_fc_okta": np.repeat(okta_fc, 24),
                         "wind_fc_ms": flat(v_fc).round(2), "free_day": np.repeat(free, 24)}, index=index)
    log = pd.Series(fault.ravel(), index=index, name="fault")
    return hist, log

HIST, FAULT_LOG = campus_history()
TRAIN_END = "2026-04-12 23:00"     # six weeks for learning, two weeks for testing
TEST_START = "2026-04-13 00:00"
# The tariff of the campus (Week 9) and its battery (Week 2)
PRICE = np.array([0.14, 0.1393, 0.1389, 0.1388, 0.1389, 0.1396, 0.1418, 0.146, 0.1525, 0.156, 0.1575, 0.158,
                  0.157, 0.1565, 0.156, 0.1555, 0.157, 0.17, 0.22, 0.24, 0.21, 0.1575, 0.1475, 0.1422])
E_MIN, E_MAX, E0, P_BAT, ETA = 50.0, 450.0, 250.0, 250.0, 0.95
S_EXPORT, K_PEAK = 0.05, 0.30       # euro/kWh exported, euro/kW of the highest import of the day
print("Setup complete. Running in Colab:", IN_COLAB, "|", len(HIST), "hours from", HIST.index[0].date(), "to",
      HIST.index[-1].date())

In [ ]:
#@title Expected answers of the exercises (run this cell; open it only after trying the exercises)
ANSWERS = {
    "Exercise 10.1": (lambda: HIST.loc['2026-04-12'].load_kw.mean(), 1e-6),
    "Exercise 10.2": (lambda: np.polyfit(HIST.loc[:TRAIN_END].between_time('18:00', '18:00').query('not free_day').temp_c, HIST.loc[:TRAIN_END].between_time('18:00', '18:00').query('not free_day').load_kw, 1)[0], 1e-6),
    "Exercise 10.3": (lambda: np.sqrt(np.mean(np.array([-12.0, 10.0, -18.0, 20.0]) ** 2)), 1e-6),
    "Exercise A": (lambda: errors(TEST.load_kw, HIST.load_kw.shift(336).loc[TEST_START:])['rmse'], 1e-6),
    "Exercise B": (lambda: errors(TEST.load_kw, fit_load(3)[0])['rmse'], 1e-6),
    "Exercise C": (lambda: float(PV_FC.loc['2026-04-20 12:00']), 1e-6),
    "Exercise D": (lambda: 3.0, 1e-9),
    "Exercise E": (lambda: STAGE10['share of the possible saving, plan with the forecast (percent)'], 1e-6),
}

## Python step 10: Fitting and forecasting

Forecasting starts with data in time order. This step cuts and averages a time series with pandas, fits a straight line through points with NumPy and measures the errors of a forecast &#91;[2](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-10/lecture.html#ref-2)&#93;.

**Started here?** If you opened the Colab notebook at this section, run the next cell once. It runs the code of the earlier sections for you, without their printouts and figures. This takes from a few seconds to a minute, and the cell does nothing when they have already run. The Colab menu Runtime, Run before, which runs every cell above the selected one, works as well.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
if "ANSWERS" in globals() and "FAULT_LOG" in globals():
    print("The earlier sections have already run. Continue with the next cell.")
else:
    import json, urllib.request
    import matplotlib.pyplot as plt
    from IPython import get_ipython
    from IPython.utils.capture import capture_output
    _cu_url = ("https://raw.githubusercontent.com/utkukose/"
               "cyber-physical-systems-NB-lecture/main/weeks/week-10/"
               "NB10_learning_from_data.ipynb")
    _cu_cells = json.load(urllib.request.urlopen(_cu_url))["cells"]
    _cu_ip, _cu_err = get_ipython(), None
    with capture_output():
        for _cu_c in _cu_cells:
            if _cu_c.get("id") == "python-step":
                break
            _cu_meta = _cu_c.get("metadata", {})
            if _cu_c["cell_type"] == "code" and _cu_meta.get("catchup", True):
                try:
                    _cu_src = _cu_ip.transform_cell("".join(_cu_c["source"]))
                    exec(_cu_src, _cu_ip.user_ns)
                except Exception as _cu_e:
                    _cu_err = _cu_e
                    break
    plt.close("all")
    if _cu_err is None:
        print("The earlier sections have run. Continue with the next cell.")
    else:
        print(f"The catch-up stopped with an error ({_cu_err}). "
              "Run the notebook from the top instead.")

### A time series with pandas

In [ ]:
# Python practice: a small example for learning the language
week = HIST.loc["2026-04-06":"2026-04-12"]        # one week, Monday to Sunday
daily = week.load_kw.resample("D").mean()           # the mean load of each day (kW)
print(daily.round(1))
print("mean load at 08:00 over the eight weeks:", round(HIST.load_kw[HIST.index.hour == 8].mean(), 1), "kW")

`HIST` is a pandas DataFrame with one row per hour and a time stamp as its index. `.loc` cuts out a period by its dates, and `resample("D").mean()` averages each day. `HIST.index.hour` gives the hour of every row, so a condition on it selects the same hour on all days.

### A straight line through points

In [ ]:
# Python practice: a small example for learning the language
mornings = HIST.loc[:TRAIN_END].between_time("08:00", "08:00")
mornings = mornings[~mornings.free_day]                       # working days only
b, a = np.polyfit(mornings.temp_c, mornings.load_kw, 1)       # load = a + b * temperature
print(f"load at 08:00 = {a:.1f} + ({b:.2f}) x temperature, from {len(mornings)} working days")

`np.polyfit(x, y, 1)` fits the line with the smallest sum of squared errors and returns the slope first. The slope is negative: The campus draws more power on a cold morning, the heating of its buildings.

### Errors of a forecast

In [ ]:
# Python practice: a small example for learning the language
actual = np.array([430.0, 465.0, 512.0])           # three hours (kW)
forecast = np.array([442.0, 455.0, 530.0])
err = actual - forecast
print("errors:", err, "| MAE", np.abs(err).mean().round(2), "| RMSE", np.sqrt((err ** 2).mean()).round(2),
      "| MAPE", (100 * np.abs(err / actual)).mean().round(2), "percent")

The mean absolute error (MAE) averages the size of the errors. The root mean square error (RMSE) squares them first, so large errors weigh more. The mean absolute percentage error (MAPE) relates each error to the actual value &#91;[3](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-10/lecture.html#ref-3)&#93;.

**Quick check.** Why is the RMSE never smaller than the MAE?

<details><summary>Answer</summary>

Squaring weighs large errors more. The root of the mean square is at least the mean of the sizes, and the two are equal only when all errors have the same size.

</details>

### Exercises for Python step 10

**Exercise 10.1.** Store the mean load of Sunday 12 April in kW in `m_101`.

In [ ]:
m_101 = None   # your answer

check("Exercise 10.1", m_101)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
m_101 = HIST.loc["2026-04-12"].load_kw.mean()

check("Exercise 10.1", m_101)

**Exercise 10.2.** Fit the line again for the load at 18:00 on working days of the six weeks and store its slope in `b_102`.

In [ ]:
b_102 = None   # your answer

check("Exercise 10.2", b_102)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
evenings = HIST.loc[:TRAIN_END].between_time("18:00", "18:00")
evenings = evenings[~evenings.free_day]
b_102 = np.polyfit(evenings.temp_c, evenings.load_kw, 1)[0]

check("Exercise 10.2", b_102)

**Exercise 10.3.** A fourth hour has the actual value 540 kW and the forecast 520 kW. Store the RMSE of the four hours in `r_103`.

In [ ]:
r_103 = None   # your answer

check("Exercise 10.3", r_103)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
err4 = np.append(err, 540.0 - 520.0)
r_103 = np.sqrt((err4 ** 2).mean())

check("Exercise 10.3", r_103)

## 1. Time series: The eight weeks of the campus &#91;[3](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-10/lecture.html#ref-3)&#93;

*Lecture: [Time series and forecast errors](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-10/lecture.html#time-series)*

**Started here?** If you opened the Colab notebook at this section, run the next cell once. It runs the code of the earlier sections for you, without their printouts and figures. This takes from a few seconds to a minute, and the cell does nothing when they have already run. The Colab menu Runtime, Run before, which runs every cell above the selected one, works as well.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
if "ANSWERS" in globals() and "mornings" in globals():
    print("The earlier sections have already run. Continue with the next cell.")
else:
    import json, urllib.request
    import matplotlib.pyplot as plt
    from IPython import get_ipython
    from IPython.utils.capture import capture_output
    _cu_url = ("https://raw.githubusercontent.com/utkukose/"
               "cyber-physical-systems-NB-lecture/main/weeks/week-10/"
               "NB10_learning_from_data.ipynb")
    _cu_cells = json.load(urllib.request.urlopen(_cu_url))["cells"]
    _cu_ip, _cu_err = get_ipython(), None
    with capture_output():
        for _cu_c in _cu_cells:
            if _cu_c.get("id") == "sec-1":
                break
            _cu_meta = _cu_c.get("metadata", {})
            if _cu_c["cell_type"] == "code" and _cu_meta.get("catchup", True):
                try:
                    _cu_src = _cu_ip.transform_cell("".join(_cu_c["source"]))
                    exec(_cu_src, _cu_ip.user_ns)
                except Exception as _cu_e:
                    _cu_err = _cu_e
                    break
    plt.close("all")
    if _cu_err is None:
        print("The earlier sections have run. Continue with the next cell.")
    else:
        print(f"The catch-up stopped with an error ({_cu_err}). "
              "Run the notebook from the top instead.")

**Why Python here.** Eight weeks of hourly measurements describe the campus: The load of its buildings, the car park, the solar plant, the wind turbine, the grid and the weather. Python splits them by time into six weeks for learning and two weeks for testing. It then measures the errors of two simple forecasts of the load.

**From the formulas to the code.** The MAE $\frac{1}{N}\sum_t |y_t - \hat{y}_t|$, the RMSE $\sqrt{\frac{1}{N}\sum_t (y_t - \hat{y}_t)^2}$ and the MAPE $\frac{100}{N}\sum_t |y_t - \hat{y}_t|/y_t$ are computed by `errors(actual, forecast)`. The forecast *yesterday* is the load shifted by 24 hours, and the forecast *last week* the load shifted by 168 hours.

In [ ]:
def errors(actual, forecast):
    """The MAE, the RMSE (both in the unit of the data) and the MAPE (percent, over the hours with an actual above 0)."""
    a, f = np.asarray(actual, float), np.asarray(forecast, float)
    e, pos = a - f, a > 0
    return dict(mae=round(float(np.abs(e).mean()), 2), rmse=round(float(np.sqrt((e ** 2).mean())), 2),
                mape=round(float(100 * np.abs(e[pos] / a[pos]).mean()), 2))

TEST = HIST.loc[TEST_START:]                                  # the test fortnight, 13 to 26 April
F_YESTERDAY = HIST.load_kw.shift(24).loc[TEST_START:]         # the same hour of the day before
F_LAST_WEEK = HIST.load_kw.shift(168).loc[TEST_START:]        # the same hour seven days before
HOLIDAY = TEST.index.normalize() == pd.Timestamp("2026-04-23")
COLD = TEST.index.normalize().isin(pd.to_datetime(["2026-04-15", "2026-04-16", "2026-04-17"]))
print("hours: training", len(HIST.loc[:TRAIN_END]), "| test", len(TEST))
for name, f in (("yesterday", F_YESTERDAY), ("last week", F_LAST_WEEK)):
    e = TEST.load_kw - f
    print(f"{name:<10} {errors(TEST.load_kw, f)} | MAE on 23 April {e[HOLIDAY].abs().mean():.2f} kW, "
          f"in the cold spell {e[COLD].abs().mean():.2f} kW")

**Python note.** `shift(24)` moves the whole column down by 24 rows, so each hour sees the value of the same hour of the day before. The first 24 rows become `NaN`, the mark of pandas for a missing value, and they lie in the training weeks.

**Exercise A.** Store the RMSE of the forecast *two weeks ago*, the load shifted by 336 hours, over the test fortnight in `rmse_a`.

In [ ]:
rmse_a = None   # your answer

check("Exercise A", rmse_a)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
rmse_a = errors(TEST.load_kw, HIST.load_kw.shift(336).loc[TEST_START:])["rmse"]

check("Exercise A", rmse_a)

In [ ]:
# Figure: the eight weeks of the campus and the forecast last week in the test fortnight
import matplotlib.dates as mdates
fig, ax = plt.subplots(2, 1, figsize=(10, 5.4))
ax[0].plot(HIST.index, HIST.load_kw, color="#1E6FA8", lw=0.7, label="load of the buildings")
ax[0].axvspan(pd.Timestamp(TEST_START), HIST.index[-1], color="#F2B33D", alpha=0.2, label="test fortnight")
ax[0].set_ylabel("load (kW)")
ax0 = ax[0].twinx()
ax0.plot(HIST.index, HIST.temp_c, color="#B53F2E", lw=0.5, alpha=0.6)
ax0.set_ylabel("temperature (C)")
ax[0].legend(fontsize=8, loc="upper right")
ax[1].plot(TEST.index, TEST.load_kw, color="#1B2430", lw=1.2, label="actual")
ax[1].plot(TEST.index, F_LAST_WEEK, color="#C98510", lw=1, label="forecast: last week")
ax[1].set_ylabel("load (kW)")
ax[1].legend(fontsize=8)
for a in ax:
    a.xaxis.set_major_formatter(mdates.DateFormatter("%d %b"))
plt.tight_layout()
plt.show()

**Observe and reflect.** On which days does the forecast *yesterday* fail most, and why? Which day breaks the forecast *last week*?

## 2. Regression: The load model of the campus &#91;[5](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-10/lecture.html#ref-5)&#93;

*Lecture: [Regression: Forecasting the load](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-10/lecture.html#regression)*

**Started here?** If you opened the Colab notebook at this section, run the next cell once. It runs the code of the earlier sections for you, without their printouts and figures. This takes from a few seconds to a minute, and the cell does nothing when they have already run. The Colab menu Runtime, Run before, which runs every cell above the selected one, works as well.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
if "ANSWERS" in globals() and "F_LAST_WEEK" in globals():
    print("The earlier sections have already run. Continue with the next cell.")
else:
    import json, urllib.request
    import matplotlib.pyplot as plt
    from IPython import get_ipython
    from IPython.utils.capture import capture_output
    _cu_url = ("https://raw.githubusercontent.com/utkukose/"
               "cyber-physical-systems-NB-lecture/main/weeks/week-10/"
               "NB10_learning_from_data.ipynb")
    _cu_cells = json.load(urllib.request.urlopen(_cu_url))["cells"]
    _cu_ip, _cu_err = get_ipython(), None
    with capture_output():
        for _cu_c in _cu_cells:
            if _cu_c.get("id") == "sec-2":
                break
            _cu_meta = _cu_c.get("metadata", {})
            if _cu_c["cell_type"] == "code" and _cu_meta.get("catchup", True):
                try:
                    _cu_src = _cu_ip.transform_cell("".join(_cu_c["source"]))
                    exec(_cu_src, _cu_ip.user_ns)
                except Exception as _cu_e:
                    _cu_err = _cu_e
                    break
    plt.close("all")
    if _cu_err is None:
        print("The earlier sections have run. Continue with the next cell.")
    else:
        print(f"The catch-up stopped with an error ({_cu_err}). "
              "Run the notebook from the top instead.")

**Why Python here.** The load depends on the hour, the type of day and the outdoor temperature. A linear regression learns this dependence from the training weeks. It then forecasts the test fortnight with the temperature forecast of the weather service.

**From the formulas to the code.** The model $\hat{y} = a_{c,h} + b\,\max(0, 15 - T)$ has one coefficient $a_{c,h}$ for each hour $h$ of each type of day $c$. One heating coefficient $b$ completes the 49 coefficients. `features` builds the matrix $X$ with one row per hour, and `np.linalg.lstsq` finds $\beta = (X^T X)^{-1} X^T y$ in `fit_load(weeks, calendar, with_temp)`.

In [ ]:
def features(df, temp_col, calendar=True, with_temp=True):
    """One row per hour: a 1 in the column of its hour and type of day, then the heating degrees max(0, 15 - T)."""
    free = df.free_day.values if calendar else (df.index.weekday >= 5)     # without the calendar: weekends only
    X = np.zeros((len(df), 49 if with_temp else 48))
    X[np.arange(len(df)), np.where(free, 24, 0) + df.index.hour] = 1
    if with_temp:
        X[:, 48] = np.maximum(0, 15 - df[temp_col].values)
    return X

def fit_load(weeks=6, calendar=True, with_temp=True):
    """Learns the model from the last weeks before the test; returns the forecast of the test, beta and the training
    errors. The training uses the measured temperature, the forecast uses the forecast of the weather service."""
    train = HIST.loc[pd.Timestamp(TEST_START) - pd.Timedelta(weeks=weeks):TRAIN_END]
    X = features(train, "temp_c", calendar, with_temp)
    beta = np.linalg.lstsq(X, train.load_kw.values, rcond=None)[0]
    forecast = features(TEST, "temp_fc_c", calendar, with_temp) @ beta
    return forecast, beta, errors(train.load_kw, X @ beta)

for with_temp in (False, True):
    for calendar in (False, True):
        for weeks in (1, 2, 6):
            f, beta, err_train = fit_load(weeks, calendar, with_temp)
            print(f"temperature {str(with_temp):<5} calendar {str(calendar):<5} {weeks} weeks: test {errors(TEST.load_kw, f)}"
                  f" | training RMSE {err_train['rmse']}")
F_MODEL, BETA, _ = fit_load(6)
print(f"heating coefficient b = {BETA[48]:.3f} kW per kelvin | a at 08:00 on a working day = {BETA[8]:.2f} kW")

**Python note.** `X[np.arange(n), cols] = 1` sets one element in each row at once, in the column of its hour and type of day. `np.linalg.lstsq` solves the least-squares problem without forming the inverse matrix, which is faster and more accurate.

**Exercise B.** Store the test RMSE of the model with the temperature and the official calendar, learnt from three weeks, in `rmse_b`.

In [ ]:
rmse_b = None   # your answer

check("Exercise B", rmse_b)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
rmse_b = errors(TEST.load_kw, fit_load(3)[0])["rmse"]

check("Exercise B", rmse_b)

Part A of the [interactive lab](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-10/lab.html) runs these forecasts in the metering room of the campus. Paste the copied rows into `MY_FORECASTS`, and the code computes each of your forecasts again.

In [ ]:
# Paste the results of lab Part A between the triple quotes.
MY_FORECASTS = """
"""
if MY_FORECASTS.strip():
    runs = pd.read_csv(io.StringIO(MY_FORECASTS.strip()))
    source = "your own forecasts in the metering room"
else:
    runs = pd.DataFrame({"run": [1, 2, 3, 4, 5], "model": ["yesterday", "last week", "regression", "temperature",
                                                          "temperature"],
                         "calendar": ["official"] * 5, "weeks": [None, None, 6, 6, 2],
                         "rmse": [83.04, 52.32, 27.86, 15.77, 18.52]})
    source = "a simulated student"
def model_rmse(r):
    if r.model == "yesterday":
        return errors(TEST.load_kw, F_YESTERDAY)["rmse"]
    if r.model == "last week":
        return errors(TEST.load_kw, F_LAST_WEEK)["rmse"]
    f = fit_load(int(r.weeks), r.calendar == "official", r.model == "temperature")[0]
    return errors(TEST.load_kw, f)["rmse"]
runs["model_rmse"] = [model_rmse(r) for r in runs.itertuples()]
print("Data:", source)
print(runs[["run", "model", "calendar", "weeks", "rmse", "model_rmse"]].to_string(index=False))

In [ ]:
# Figure: the test error against the length of the training and the forecast of the fortnight
import matplotlib.dates as mdates
fig, ax = plt.subplots(1, 2, figsize=(10, 3.4))
for with_temp, col in ((False, "#C98510"), (True, "#1E6FA8")):
    rm = [errors(TEST.load_kw, fit_load(w, True, with_temp)[0])["rmse"] for w in range(1, 7)]
    ax[0].plot(range(1, 7), rm, "o-", color=col, label="with temperature" if with_temp else "hour and day only")
ax[0].set_xlabel("weeks of training")
ax[0].set_ylabel("test RMSE (kW)")
ax[0].legend(fontsize=8)
ax[1].plot(TEST.index, TEST.load_kw, color="#1B2430", lw=1.2, label="actual")
ax[1].plot(TEST.index, F_MODEL, color="#1E6FA8", lw=1, label="regression, six weeks")
ax[1].set_ylabel("load (kW)")
ax[1].legend(fontsize=8)
ax[1].xaxis.set_major_formatter(mdates.DateFormatter("%d %b"))
plt.tight_layout()
plt.show()

**Observe and reflect.** Why do longer windows help the model with temperature but hurt the model without it? What does the model do on 23 April without the official calendar?

## 3. Forecasting the generation: The solar plant and the wind turbine &#91;[7](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-10/lecture.html#ref-7)&#93;

*Lecture: [Forecasting the generation](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-10/lecture.html#generation)*

**Started here?** If you opened the Colab notebook at this section, run the next cell once. It runs the code of the earlier sections for you, without their printouts and figures. This takes from a few seconds to a minute, and the cell does nothing when they have already run. The Colab menu Runtime, Run before, which runs every cell above the selected one, works as well.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
if "ANSWERS" in globals() and "fit_load" in globals():
    print("The earlier sections have already run. Continue with the next cell.")
else:
    import json, urllib.request
    import matplotlib.pyplot as plt
    from IPython import get_ipython
    from IPython.utils.capture import capture_output
    _cu_url = ("https://raw.githubusercontent.com/utkukose/"
               "cyber-physical-systems-NB-lecture/main/weeks/week-10/"
               "NB10_learning_from_data.ipynb")
    _cu_cells = json.load(urllib.request.urlopen(_cu_url))["cells"]
    _cu_ip, _cu_err = get_ipython(), None
    with capture_output():
        for _cu_c in _cu_cells:
            if _cu_c.get("id") == "sec-3":
                break
            _cu_meta = _cu_c.get("metadata", {})
            if _cu_c["cell_type"] == "code" and _cu_meta.get("catchup", True):
                try:
                    _cu_src = _cu_ip.transform_cell("".join(_cu_c["source"]))
                    exec(_cu_src, _cu_ip.user_ns)
                except Exception as _cu_e:
                    _cu_err = _cu_e
                    break
    plt.close("all")
    if _cu_err is None:
        print("The earlier sections have run. Continue with the next cell.")
    else:
        print(f"The catch-up stopped with an error ({_cu_err}). "
              "Run the notebook from the top instead.")

**Why Python here.** The solar plant and the wind turbine follow the weather. The weather service forecasts the cloud cover and the wind speed of the next day, and the models of Week 1 turn them into power.

**From the formulas to the code.** The cloud factor $1 - 0.75\,(N/8)^{3.4}$ of Kasten and Czeplak scales the clear-sky irradiance for a cloud cover of $N$ oktas in `cloud_factor`. `pv_power(irr, temp)` is the solar plant and `wind_power(v)` the power curve of the turbine, both from Week 1.

In [ ]:
PV_FC = pv_power(HIST.irr_clear_wm2 * cloud_factor(HIST.cloud_fc_okta), HIST.temp_fc_c)   # the cloud forecast
PV_CLEAR = pv_power(HIST.irr_clear_wm2, HIST.temp_fc_c)                                     # always a clear sky
PV_YESTERDAY = HIST.pv_kw.shift(24)
WIND_FC = pd.Series(wind_power(HIST.wind_fc_ms.values), index=HIST.index)
ok = ~FAULT_LOG & (HIST.index >= "2026-03-03")           # from the second day on, without the hours of the log
for name, f in (("yesterday", PV_YESTERDAY), ("clear sky", PV_CLEAR), ("cloud forecast", PV_FC)):
    err = errors(HIST.pv_kw[ok], f[ok])
    print(f"solar plant, {name:<15} MAE {err['mae']:6.2f} kW, RMSE {err['rmse']:6.2f} kW, "
          f"bias {(HIST.pv_kw - f)[ok].mean():6.2f} kW")
for name, f in (("power curve of the forecast", WIND_FC), ("yesterday", HIST.wind_kw.shift(24))):
    err = errors(HIST.wind_kw[24:], f[24:])
    print(f"wind turbine, {name:<28} MAE {err['mae']:6.2f} kW, RMSE {err['rmse']:6.2f} kW")

**Python note.** `FAULT_LOG` is a Series of booleans with the same index as `HIST`. `~` negates it, and `&` combines two conditions hour by hour. The code prints only the MAE and the RMSE: At dawn and at dusk the actual power is small, and a percentage of it says little.

**Exercise C.** Store the forecast of the solar plant in kW for 12:00 on Monday 20 April in `pv_c`.

In [ ]:
pv_c = None   # your answer

check("Exercise C", pv_c)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
pv_c = float(PV_FC.loc["2026-04-20 12:00"])

check("Exercise C", pv_c)

In [ ]:
# Figure: the forecasts of the solar plant on four days of the training weeks
fig, axes = plt.subplots(1, 4, figsize=(11, 2.8), sharey=True)
for ax, day in zip(axes, ["2026-04-01", "2026-04-02", "2026-04-06", "2026-04-07"]):
    sl = slice(day + " 05:00", day + " 20:00")
    ax.plot(HIST.loc[sl].index.hour, HIST.pv_kw.loc[sl], color="#1B2430", lw=1.4, label="actual")
    ax.plot(HIST.loc[sl].index.hour, PV_FC.loc[sl], color="#1E6FA8", lw=1.2, label="cloud forecast")
    ax.plot(HIST.loc[sl].index.hour, PV_YESTERDAY.loc[sl], color="#C98510", lw=1, ls="--", label="yesterday")
    ax.set_title(f"{pd.Timestamp(day):%a %d %b}, forecast {HIST.cloud_fc_okta.loc[day + ' 12:00']:.0f} oktas", fontsize=8)
    ax.set_xlabel("hour")
axes[0].set_ylabel("solar plant (kW)")
axes[0].legend(fontsize=7)
plt.tight_layout()
plt.show()

**Observe and reflect.** On which kind of day does the cloud forecast help most, and when does it mislead?

## 4. Anomaly detection: The faults of the solar plant &#91;[9](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-10/lecture.html#ref-9)&#93;

*Lecture: [Anomaly detection](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-10/lecture.html#anomaly-detection)*

**Started here?** If you opened the Colab notebook at this section, run the next cell once. It runs the code of the earlier sections for you, without their printouts and figures. This takes from a few seconds to a minute, and the cell does nothing when they have already run. The Colab menu Runtime, Run before, which runs every cell above the selected one, works as well.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
if "ANSWERS" in globals() and "PV_FC" in globals():
    print("The earlier sections have already run. Continue with the next cell.")
else:
    import json, urllib.request
    import matplotlib.pyplot as plt
    from IPython import get_ipython
    from IPython.utils.capture import capture_output
    _cu_url = ("https://raw.githubusercontent.com/utkukose/"
               "cyber-physical-systems-NB-lecture/main/weeks/week-10/"
               "NB10_learning_from_data.ipynb")
    _cu_cells = json.load(urllib.request.urlopen(_cu_url))["cells"]
    _cu_ip, _cu_err = get_ipython(), None
    with capture_output():
        for _cu_c in _cu_cells:
            if _cu_c.get("id") == "sec-4":
                break
            _cu_meta = _cu_c.get("metadata", {})
            if _cu_c["cell_type"] == "code" and _cu_meta.get("catchup", True):
                try:
                    _cu_src = _cu_ip.transform_cell("".join(_cu_c["source"]))
                    exec(_cu_src, _cu_ip.user_ns)
                except Exception as _cu_e:
                    _cu_err = _cu_e
                    break
    plt.close("all")
    if _cu_err is None:
        print("The earlier sections have run. Continue with the next cell.")
    else:
        print(f"The catch-up stopped with an error ({_cu_err}). "
              "Run the notebook from the top instead.")

**Why Python here.** The pyranometer measures the irradiance, so the plant should deliver what the model of Week 1 expects. Python compares the meter with this expectation and standardises the residual with its spread in the training weeks. It then raises alarms with a threshold and with the cumulative sum (CUSUM), and it compares them with the maintenance log.

**From the formulas to the code.** The relative residual $e_t = P_t/\hat{P}_t - 1$ with $\hat{P}_t$ = `pv_power(irradiance, temperature)` is divided by its standard deviation $\sigma$ in the training weeks: $z_t = e_t/\sigma$. The threshold alarms when $|z_t| > k$. The CUSUM adds $S^+_t = \max(0, S^+_{t-1} + z_t - \kappa)$ and $S^-_t = \min(0, S^-_{t-1} + z_t + \kappa)$, alarms when $S^+_t > h$ or $S^-_t < -h$ and then starts again at zero. `score` compares the alarms with `FAULT_LOG`.

In [ ]:
DAYLIGHT = HIST.irr_wm2 >= 100                                         # hours with enough sun for the model
REL = (HIST.pv_kw / pv_power(HIST.irr_wm2, HIST.temp_c) - 1).where(DAYLIGHT)
SIGMA = REL.loc[:TRAIN_END].std()                                        # the spread of a healthy plant
Z = (REL / SIGMA).loc[TEST_START:]
print(f"sigma = {100 * SIGMA:.3f} percent from {int(DAYLIGHT.loc[:TRAIN_END].sum())} daylight hours of training")

def threshold_alarms(k):
    return (Z.abs() > k) & DAYLIGHT.loc[TEST_START:]

def cusum_alarms(kappa=0.5, h=4.0):
    sp = sm = 0.0
    alarms = pd.Series(False, index=Z.index)
    for t, z in Z.items():
        if np.isnan(z):
            continue                                                     # night: the sums keep their values
        sp, sm = max(0.0, sp + z - kappa), min(0.0, sm + z + kappa)
        if sp > h or sm < -h:
            alarms[t] = True
            sp = sm = 0.0                                                # the alarm is acknowledged
    return alarms

FAULTS = {"inverter trip": ("2026-04-14 10:00", "2026-04-14 13:00"),
          "frozen meter": ("2026-04-18 13:00", "2026-04-18 16:00"),
          "dust": ("2026-04-20 00:00", "2026-04-26 23:00")}

def score(alarms):
    """Counts the hours of the test fortnight with daylight and compares the alarms with the maintenance log."""
    day, fault = DAYLIGHT.loc[TEST_START:], FAULT_LOG.loc[TEST_START:]
    a = alarms & day
    tp, fp = int((a & fault).sum()), int((a & ~fault).sum())
    fn, tn = int((~a & fault & day).sum()), int((~a & ~fault & day).sum())
    first = {}
    for name, (start, end) in FAULTS.items():
        hits = a.loc[start:end][a.loc[start:end]]
        first[name] = hits.index[0].strftime("%d %b %H:%M") if len(hits) else "missed"
    return dict(tp=tp, fp=fp, fn=fn, tn=tn, precision=round(tp / max(1, tp + fp), 3),
                recall=round(tp / max(1, tp + fn), 3), first=first)

for k in (2.0, 2.5, 3.0, 3.5):
    print(f"threshold k = {k}:", score(threshold_alarms(k)))
print("CUSUM kappa = 0.5, h = 4:", score(cusum_alarms(0.5, 4.0)))

**Python note.** A `for` loop over `Z.items()` gives each time stamp with its value. The CUSUM needs this loop, because each sum depends on the one of the hour before. `hits.index[0]` is the time of the first alarm within a fault.

**Exercise D.** Store the smallest threshold $k$ among 2, 2.5, 3, 3.5 and 4 that raises no false alarm in the test fortnight in `k_d`.

In [ ]:
k_d = None   # your answer

check("Exercise D", k_d)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
k_d = next(k for k in (2.0, 2.5, 3.0, 3.5, 4.0) if score(threshold_alarms(k))["fp"] == 0)

check("Exercise D", k_d)

Part C of the [interactive lab](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-10/lab.html) runs the detectors at the solar plant of the campus. Paste the copied rows into `MY_ALARMS`, and the code runs each of your detectors again.

In [ ]:
# Paste the results of lab Part C between the triple quotes.
MY_ALARMS = """
"""
if MY_ALARMS.strip():
    runs = pd.read_csv(io.StringIO(MY_ALARMS.strip()))
    source = "your own runs at the solar plant"
else:
    runs = pd.DataFrame({"run": [1, 2, 3, 4], "detector": ["threshold", "threshold", "threshold", "cusum"],
                         "k": [2.0, 3.0, 2.5, None], "kappa": [None, None, None, 0.5], "h": [None, None, None, 4.0],
                         "false_alarms": [4, 0, 1, 0]})
    source = "a simulated student"
def model_fp(r):
    alarms = threshold_alarms(r.k) if r.detector == "threshold" else cusum_alarms(r.kappa, r.h)
    return score(alarms)["fp"]
runs["model_false_alarms"] = [model_fp(r) for r in runs.itertuples()]
print("Data:", source)
print(runs[["run", "detector", "k", "kappa", "h", "false_alarms", "model_false_alarms"]].to_string(index=False))

In [ ]:
# Figure: the standardised residual of the fortnight with the faults and the alarms
fig, ax = plt.subplots(2, 1, figsize=(10, 4.6), sharex=True)
for start, end in FAULTS.values():
    for a in ax:
        a.axvspan(pd.Timestamp(start), pd.Timestamp(end) + pd.Timedelta(hours=1), color="#F2B33D", alpha=0.25)
ax[0].plot(Z.index, Z.clip(-12, 12), ".", ms=3, color="#1E6FA8")
ax[0].axhline(3, color="#B53F2E", ls="--", lw=1)
ax[0].axhline(-3, color="#B53F2E", ls="--", lw=1)
ax[0].set_ylabel("z (clipped at 12)")
al = cusum_alarms(0.5, 4.0)
ax[1].plot(Z.index, threshold_alarms(3.0).astype(int) + 0.02, "|", color="#B53F2E", ms=10, label="threshold k = 3")
ax[1].plot(Z.index, al.astype(int) * 0.6, "|", color="#2E8B57", ms=10, label="CUSUM 0.5 / 4")
ax[1].set_ylim(0.3, 1.2)
ax[1].set_yticks([])
ax[1].legend(fontsize=8, loc="upper left")
ax[1].xaxis.set_major_formatter(mdates.DateFormatter("%d %b"))
plt.tight_layout()
plt.show()

**Observe and reflect.** Which fault does the threshold catch at once, and which one does only the CUSUM catch early? Why does the frozen meter give a positive residual?

## 5. The campus microgrid, stage 10

*Lecture: [The campus microgrid, stage 10: Forecasts of load and generation](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-10/lecture.html#the-campus-microgrid-stage-10)*

**Started here?** If you opened the Colab notebook at this section, run the next cell once. It runs the code of the earlier sections for you, without their printouts and figures. This takes from a few seconds to a minute, and the cell does nothing when they have already run. The Colab menu Runtime, Run before, which runs every cell above the selected one, works as well.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
if "ANSWERS" in globals() and "cusum_alarms" in globals():
    print("The earlier sections have already run. Continue with the next cell.")
else:
    import json, urllib.request
    import matplotlib.pyplot as plt
    from IPython import get_ipython
    from IPython.utils.capture import capture_output
    _cu_url = ("https://raw.githubusercontent.com/utkukose/"
               "cyber-physical-systems-NB-lecture/main/weeks/week-10/"
               "NB10_learning_from_data.ipynb")
    _cu_cells = json.load(urllib.request.urlopen(_cu_url))["cells"]
    _cu_ip, _cu_err = get_ipython(), None
    with capture_output():
        for _cu_c in _cu_cells:
            if _cu_c.get("id") == "sec-5":
                break
            _cu_meta = _cu_c.get("metadata", {})
            if _cu_c["cell_type"] == "code" and _cu_meta.get("catchup", True):
                try:
                    _cu_src = _cu_ip.transform_cell("".join(_cu_c["source"]))
                    exec(_cu_src, _cu_ip.user_ns)
                except Exception as _cu_e:
                    _cu_err = _cu_e
                    break
    plt.close("all")
    if _cu_err is None:
        print("The earlier sections have run. Continue with the next cell.")
    else:
        print(f"The catch-up stopped with an error ({_cu_err}). "
              "Run the notebook from the top instead.")

**Why Python here.** Stage 10 forecasts the import of the campus for each day of the test fortnight and plans the battery of Week 9 with it. The plan then meets the actual day, and the cost shows what a forecast is worth.

In [ ]:
def plan_battery(net):
    """The battery schedule of Week 9 for one day (net: import without the battery, kW); returns its power, + discharge."""
    T = 24
    C, D, G, X, E, PK = 0, T, 2 * T, 3 * T, 4 * T, 5 * T
    cost = np.zeros(5 * T + 1)
    cost[G:G + T], cost[X:X + T], cost[PK] = PRICE, -S_EXPORT, K_PEAK
    A_eq, b_eq = np.zeros((2 * T, 5 * T + 1)), np.zeros(2 * T)
    A_ub, b_ub = np.zeros((T + 1, 5 * T + 1)), np.zeros(T + 1)
    for t in range(T):
        A_eq[t, [G + t, X + t, C + t, D + t]] = [1, -1, -1, 1]          # the balance: g - x - c + d = net
        b_eq[t] = net[t]
        A_eq[T + t, [E + t, C + t, D + t]] = [1, -ETA, 1 / ETA]         # the storage
        if t > 0:
            A_eq[T + t, E + t - 1] = -1
        else:
            b_eq[T] = E0
        A_ub[t, [G + t, PK]] = [1, -1]                                  # the peak of the day
    A_ub[T, E + T - 1], b_ub[T] = -1, -E0                               # end with the energy of the start
    bounds = [(0, P_BAT)] * 2 * T + [(0, None)] * 2 * T + [(E_MIN, E_MAX)] * T + [(0, None)]
    x = linprog(cost, A_ub=A_ub, b_ub=b_ub, A_eq=A_eq, b_eq=b_eq, bounds=bounds, method="highs").x
    return x[D:D + T] - x[C:C + T]

def day_cost(grid):
    """The cost of a day (euro): energy at the price of the hour, exports at 0.05 euro/kWh and the peak charge."""
    grid = np.asarray(grid, float)
    return float(PRICE @ grid.clip(min=0) - S_EXPORT * (-grid).clip(min=0).sum() + K_PEAK * max(0.0, grid.max()))

NET = TEST.grid_kw.values                                                       # the actual import
NET_FC = F_MODEL + TEST.ev_kw.values - PV_FC.loc[TEST_START:].values - WIND_FC.loc[TEST_START:].values
plans = {"without the battery": None, "perfect knowledge": NET, "forecast of stage 10": NET_FC,
         "last week": HIST.grid_kw.shift(168).loc[TEST_START:].values,
         "yesterday": HIST.grid_kw.shift(24).loc[TEST_START:].values}
COSTS, PEAKS = {}, {}
for name, guess in plans.items():
    cost, peaks = 0.0, []
    for k in range(14):
        day = slice(24 * k, 24 * k + 24)
        grid = NET[day] if guess is None else NET[day] - plan_battery(guess[day])   # the plan meets the actual day
        cost += day_cost(grid)
        peaks.append(grid.max())
    COSTS[name], PEAKS[name] = round(cost, 2), round(float(np.mean(peaks)), 1)

possible = COSTS["without the battery"] - COSTS["perfect knowledge"]
STAGE10 = {
    "load, regression with temperature and calendar: RMSE (kW)": errors(TEST.load_kw, F_MODEL)["rmse"],
    "load, regression: MAPE (percent)": errors(TEST.load_kw, F_MODEL)["mape"],
    "load, last week: RMSE (kW)": errors(TEST.load_kw, F_LAST_WEEK)["rmse"],
    "solar plant, cloud forecast: RMSE (kW)": errors(HIST.pv_kw[ok], PV_FC[ok])["rmse"],
    "import, forecast of stage 10: RMSE (kW)": errors(NET, NET_FC)["rmse"],
    "import, last week: RMSE (kW)": errors(NET, plans["last week"])["rmse"],
    "fortnight without the battery (euro)": COSTS["without the battery"],
    "fortnight, plan with perfect knowledge (euro)": COSTS["perfect knowledge"],
    "fortnight, plan with the forecast (euro)": COSTS["forecast of stage 10"],
    "fortnight, plan with last week (euro)": COSTS["last week"],
    "fortnight, plan with yesterday (euro)": COSTS["yesterday"],
    "share of the possible saving, plan with the forecast (percent)":
        round(100 * (COSTS["without the battery"] - COSTS["forecast of stage 10"]) / possible, 2),
    "share of the possible saving, plan with last week (percent)":
        round(100 * (COSTS["without the battery"] - COSTS["last week"]) / possible, 2),
    "faults found by the CUSUM (kappa 0.5, h 4)": sum(v != "missed" for v in score(cusum_alarms())["first"].values()),
    "false alarms of the CUSUM (kappa 0.5, h 4)": score(cusum_alarms())["fp"],
}
for key, value in STAGE10.items():
    print(f"{key:<66} {value}")
print("mean daily peak of the import (kW):", PEAKS)

**Python note.** The plan is made at midnight with a guess of the import and then applied to the actual day: $g_t = n_t - p_t$. The battery does exactly what the plan says, whatever the day brings.

In [ ]:
# Figure: the cost of the fortnight with each plan and the cold Thursday 16 April
fig, ax = plt.subplots(1, 2, figsize=(10, 3.4))
names = list(COSTS)
ax[0].barh(names, [COSTS[n] for n in names], color=["#56616D", "#2E8B57", "#1E6FA8", "#C98510", "#B53F2E"])
ax[0].set_xlim(17000, 18200)
ax[0].set_xlabel("cost of the fortnight (euro)")
day = slice(24 * 3, 24 * 4)
ax[1].step(HOURS, NET[day], where="post", color="#56616D", label="actual import without the battery")
ax[1].step(HOURS, NET_FC[day], where="post", color="#1E6FA8", ls="--", label="forecast of the import")
ax[1].step(HOURS, NET[day] - plan_battery(NET_FC[day]), where="post", color="#1E6FA8", lw=2,
           label="import with the plan")
ax[1].set_xlabel("hour of 16 April")
ax[1].set_ylabel("import (kW)")
ax[1].legend(fontsize=7)
plt.tight_layout()
plt.show()

**Exercise E.** Store the share in percent of the possible saving that the plan with the forecast achieves, as kept in `STAGE10`, in `pct_e`.

In [ ]:
pct_e = None   # your answer

check("Exercise E", pct_e)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
pct_e = STAGE10["share of the possible saving, plan with the forecast (percent)"]

check("Exercise E", pct_e)

**Observe and reflect.** The forecast of the import still has an RMSE of more than 40 kW. Why does the plan with it keep most of the possible saving?

## 6. Your application: The application switch

*Lecture: [Your application](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-10/lecture.html#your-application)*

**Started here?** If you opened the Colab notebook at this section, run the next cell once. It runs the code of the earlier sections for you, without their printouts and figures. This takes from a few seconds to a minute, and the cell does nothing when they have already run. The Colab menu Runtime, Run before, which runs every cell above the selected one, works as well.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
if "ANSWERS" in globals() and "STAGE10" in globals():
    print("The earlier sections have already run. Continue with the next cell.")
else:
    import json, urllib.request
    import matplotlib.pyplot as plt
    from IPython import get_ipython
    from IPython.utils.capture import capture_output
    _cu_url = ("https://raw.githubusercontent.com/utkukose/"
               "cyber-physical-systems-NB-lecture/main/weeks/week-10/"
               "NB10_learning_from_data.ipynb")
    _cu_cells = json.load(urllib.request.urlopen(_cu_url))["cells"]
    _cu_ip, _cu_err = get_ipython(), None
    with capture_output():
        for _cu_c in _cu_cells:
            if _cu_c.get("id") == "sec-6":
                break
            _cu_meta = _cu_c.get("metadata", {})
            if _cu_c["cell_type"] == "code" and _cu_meta.get("catchup", True):
                try:
                    _cu_src = _cu_ip.transform_cell("".join(_cu_c["source"]))
                    exec(_cu_src, _cu_ip.user_ns)
                except Exception as _cu_e:
                    _cu_err = _cu_e
                    break
    plt.close("all")
    if _cu_err is None:
        print("The earlier sections have run. Continue with the next cell.")
    else:
        print(f"The catch-up stopped with an error ({_cu_err}). "
              "Run the notebook from the top instead.")

**Why Python here.** Wherever a quantity follows the hour, the type of day and the weather, the same regression forecasts it and the same residual finds what does not fit. The function of this section learns five weeks of a series of any kind, forecasts the sixth and flags the hours with a residual above three standard deviations.

The series and their numbers are illustrative, each with one unusual event in the last week. Set `APP` to one of the six names and run the cell.

In [ ]:
APP = "district_heat"
# choose: bakery, water, bus_line, web_server, district_heat, emergency
APPS = {  # description, unit, profile over the hours of a working day, level on free days, effect per kelvin below 15 C
    "bakery": ("loaves sold by a bakery chain", "loaves per hour", 20 + 180 * np.exp(-((HOURS - 8) / 1.6) ** 2)
               + 110 * np.exp(-((HOURS - 18) / 1.8) ** 2), 1.3, 0.5),
    "water": ("the water demand of a town", "m3 per hour", 220 + 180 * np.sin(np.pi * np.clip(HOURS - 5, 0, 17) / 17), 1.05, 1.0),
    "bus_line": ("passengers of a city bus line", "passengers per hour", 10 + 160 * np.exp(-((HOURS - 8) / 1.3) ** 2)
                 + 140 * np.exp(-((HOURS - 17.5) / 1.6) ** 2) + 40 * (HOURS >= 6) * (HOURS <= 21), 0.45, 1.5),
    "web_server": ("requests to a university web server", "thousand requests per hour",
                   15 + 85 * np.exp(-((HOURS - 14) / 4.5) ** 2), 0.6, 0.0),
    "district_heat": ("the heat demand of a district heating network", "MW", 18 + 8 * (HOURS >= 6) * (HOURS <= 22), 0.95, 1.8),
    "emergency": ("patients arriving at an emergency department", "patients per hour", 3 + 9 * np.exp(-((HOURS - 19) / 4.0) ** 2),
                  1.2, 0.08)}
desc, unit, profile, free_level, per_kelvin = APPS[APP]
rng = np.random.default_rng(3)
idx = pd.date_range("2026-01-05", periods=42 * 24, freq="h")
df = pd.DataFrame(index=idx)
df["free_day"] = idx.weekday >= 5
df["temp_c"] = (6 + 5 * np.sin(np.pi * (idx.hour - 9) / 12) + np.repeat(rng.normal(0, 4, 42), 24)).round(1)
df["temp_fc_c"] = (df.temp_c + np.repeat(rng.normal(0, 1, 42), 24)).round(1)
level = np.where(df.free_day, free_level, 1.0) * profile[idx.hour]
df["value"] = (level + per_kelvin * np.maximum(0, 15 - df.temp_c)) * (1 + 0.04 * rng.normal(0, 1, len(df)))
event = slice("2026-02-11 09:00", "2026-02-11 14:00")
df.loc[event, "value"] *= 1.5                                    # the unusual event of the last week
train, test = df.loc[:"2026-02-08 23:00"], df.loc["2026-02-09":]
beta = np.linalg.lstsq(features(train, "temp_c"), train.value.values, rcond=None)[0]
sigma = np.std(train.value.values - features(train, "temp_c") @ beta)
forecast = features(test, "temp_fc_c") @ beta
z = (test.value.values - forecast) / sigma
print(f"{APP}: {desc} ({unit})")
print("  forecast of the last week:", errors(test.value, forecast))
print("  hours with |z| > 3:", [t.strftime("%a %d %b %H:%M") for t in test.index[np.abs(z) > 3]])

**Python note.** `features` from Section 2 works for any DataFrame with the columns `free_day` and a temperature, so the same model serves all six series. The event of Wednesday 11 February lies in the last week, and the alarms show whether the residual finds it.

**Observe and reflect.** For which series does the regression forecast best, and for which one does the residual find the event most clearly? What would you add to the model of your own series?

## Application challenges

Each challenge transfers the ideas of the week to a field of computing. Choose the one closest to your interests and solve it in a copy of this Colab notebook.

| Field | Challenge |
|---|---|
| Software engineering | Wrap the load model in a class with the methods `fit`, `predict` and `score`, and test that it reproduces the numbers of Section 2. |
| Embedded systems | The edge computer of the metering room has no NumPy. Write the regression with plain Python lists, using that each coefficient of an hour is a mean when the temperature is left out. |
| Internet of Things | One smart meter loses six hours of data on 15 April. Fill the gap in three ways and measure which one harms the forecast least. |
| Computer networks | The weather forecast arrives late on two days. Forecast these days with the temperature of the day before and measure the cost of the delay. |
| Control engineering | Plan the battery again every six hours with the measured import so far, as model predictive control does, and compare the cost with the plan of midnight. |
| Data science | Add the temperature of the hour before and the load of the same hour a week earlier as features, and test whether the RMSE falls. |
| Cybersecurity | An attacker adds 3 percent to the irradiance measured by the pyranometer. Which faults does the CUSUM still find, and what does the attack hide? |
| Artificial intelligence | Train a small neural network on the same features and compare it with the regression on the test fortnight, the holiday and the cold spell. |

## After the Colab notebook

The self-assessment and the reflection are in the [interactive lab](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-10/lab.html), tabs *Check yourself* and *Reflect and export*. The weekly task and the research assignment are on the [week overview](https://github.com/utkukose/cyber-physical-systems-NB-lecture/tree/main/weeks/week-10).

---

<div align="center">

**Prof. Dr. Utku Kose**

Full Professor, Department of Computer Engineering, Süleyman Demirel University, Isparta, Türkiye  
Founding Director, AI Application and Research Center (YAZEM), Süleyman Demirel University  
Head of the Computer Science Division, Department of Computer Engineering, Süleyman Demirel University  
Additional affiliations: University of North Dakota (USA), Universidad Panamericana (Mexico City, Mexico), Vel Tech University (Chennai, India)  
IEEE Senior Member, ACM Professional Member

[ORCID 0000-0002-9652-6415](https://orcid.org/0000-0002-9652-6415) | [utkukose.com](https://www.utkukose.com) | [github.com/utkukose](https://github.com/utkukose)

[utkukose@sdu.edu.tr](mailto:utkukose@sdu.edu.tr) | [utku.kose@und.edu](mailto:utku.kose@und.edu) | [ukose@up.edu.mx](mailto:ukose@up.edu.mx) | [utkukose@gmail.com](mailto:utkukose@gmail.com)

This course is updated in line with current developments in the field. First version: November 2026.

</div>